In [1]:
# Google Colab Notebook: train_predict_success_model.ipynb
# Цель: обучить модель на исторических данных и предсказывать вероятность успеха для новой команды

import pandas as pd
import numpy as np
import joblib
import json
import re
import nltk
import pymorphy2
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

nltk.download('punkt')
nltk.download('stopwords')
from nltk.corpus import stopwords

# Загрузка данных
df = pd.read_csv('/content/employee_feedback.csv')

# Предобработка текста
morph = pymorphy2.MorphAnalyzer()
stop_words = set(stopwords.words('russian'))

def preprocess(text):
    text = re.sub(r"[^\w\s]", "", str(text).lower())
    tokens = nltk.word_tokenize(text)
    lemmatized = [morph.parse(w)[0].normal_form for w in tokens if w not in stop_words]
    return " ".join(lemmatized)

df['cleaned'] = df['Текст_отзыва'].apply(preprocess)

# Векторизация
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['cleaned'])
y = df['Успешность']

# Обучение модели
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LogisticRegression()
model.fit(X_train, y_train)

# Оценка точности
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

# Сохранение модели и векторайзера
joblib.dump(model, 'success_predictor.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')

# Предсказание для всей команды (новые данные)
X_team = vectorizer.transform(df['cleaned'])
pred_probs = model.predict_proba(X_team)[:, 1]
success_rate = np.mean(pred_probs)

# Генерация результатов
result = {
    "team_success_rate": round(float(success_rate), 2),
    "positives": ["Командный дух", "Фокус на цели"],  # Заглушки
    "negatives": ["Недостаток планирования"]
}

with open("analysis_result.json", "w", encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("Success rate:", success_rate)
print("Результат сохранён в analysis_result.json")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\shuma\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\shuma\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


FileNotFoundError: [Errno 2] No such file or directory: '/content/employee_feedback.csv'